# main — run the three maps on the Qiskit grid and store the result

This notebook **computes**; it draws nothing. It propagates the FMO complex three ways —

* **Lindblad** (Markovian): one $16\times16$ matrix exponential $\to M_k(\Delta t)$, applied as an
  operator sum of dilated circuits;
* **HEOM** (non-Markovian): one matrix exponential of the extended ADO generator $\to$ short-time maps
  $\to$ transfer tensors $\to$ one dilated circuit;
* **path integral** (non-Markovian): TEMPO/TNPI short-time maps $\to$ the same route;

benchmarks each against the ordinary qutip solver, and writes everything to
`data/run_N{N}.npz`. **`results.ipynb` reads that file and only plots.**

Set `N = 4` or `N = 7` below. Because $D=N^2$ enters the register size, the HEOM hierarchy depth and
the TEMPO bond dimension quadratically, the 7-site case needs its own (smaller) settings — see the
configuration cell.

### Imports

In [5]:
import os, time
os.makedirs('data', exist_ok=True)

import numpy as np

# .ipynb files cannot be imported by Python on their own; import_ipynb registers
# the loader that makes `from grid import ...` work exactly like it would for a .py
import import_ipynb
from grid import vec, unvec
from heom_map import heom_onestep_propagator, heom_shorttime_maps, standard_heom_rho
from lindblad_map import lindblad_onestep_map, standard_lindblad_rho
from pathintegral_map import PathIntegralMap
from circuit import populations_kraus_qiskit, populations_memory_qiskit
from validation_and_helpers import (populations_enhanced, populations_enhanced_memory,
                                    save_run)

# None of the imported notebooks defines a model parameter or a physical constant --
# they all take them as arguments, so the values set below are the only ones in play.

### Global parameters

In [ ]:
# ---- physical constants (energies in cm^-1, time tau = 2*pi*c*t in cm) ----
_C_CM = 2.99792458e10                    # speed of light [cm/s]
FS_TO_CM = 1e-15 * 2 * np.pi * _C_CM     # tau[cm] per t[fs]
KB_CM = 0.6950348004                     # Boltzmann constant [cm^-1/K]

N = 4                                    # 4-site (Seneviratne eq. 30) or 7-site FMO

if N == 4:
    H_FMO = np.array([[12375.0, -87.7,   5.5,  -5.9],
                      [-87.7, 12495.0,  30.8,   8.2],
                      [5.5,      30.8, 12175.0, -53.4],
                      [-5.9,      8.2, -53.4, 12285.0]])
elif N == 7:
    H_FMO = np.array([[1410.0, -87.7, 5.5, -5.9, 6.7, -13.7, -9.9],
                      [-87.7, 1530.0, 30.8, 8.2, 0.7, 11.8, 4.3],
                      [5.5, 30.8, 1210.0, -53.5, -2.2, -9.6, 6.0],
                      [-5.9, 8.2, -53.5, 1320.0, -70.7, -17.0, -63.3],
                      [6.7, 0.7, -2.2, -70.7, 1480.0, 81.1, -1.3],
                      [-13.7, 11.8, -9.6, -17.0, 81.1, 1630.0, 39.7],
                      [-9.9, 4.3, 6.0, -63.3, -1.3, 39.7, 1440.0]])
else:
    raise ValueError('N must be 4 or 7')

H_FMO = H_FMO - np.trace(H_FMO) / N * np.eye(N)
LAM_FMO, GAM_FMO, T_FMO = 35.0, 106.18, 300.0

# Bundled once and splatted into every method below, so HEOM and Lindblad are
# guaranteed to run on identical parameters.
MODEL = dict(H=H_FMO, lam=LAM_FMO, gamma=GAM_FMO, T_K=T_FMO,
             KB_CM=KB_CM, FS_TO_CM=FS_TO_CM)

# ---- run configuration ---------------------------------------------------
d, dt_fs = N, 10.0                 # Systemsize: N-site, time step = 10.0 fs
rho0 = np.zeros((d, d), complex); rho0[0, 0] = 1.0   # excitation starts at site 0
t_fs = np.arange(0.0, 1001.0, dt_fs)                 # 0..1000 fs
n_steps = len(t_fs) - 1

# D = N^2 enters everything quadratically, so the 7-site case needs smaller
# settings to stay tractable (measured on this machine):
#   HEOM depth 3 at N=7 -> 680 ADOs -> a 33320^2 generator = 17.8 GB  (impossible)
#   HEOM depth 2 at N=7 -> 120 ADOs -> a  5880^2 generator, expm ~ 100 s  (fine)
#   TEMPO chi_max 512 at N=7 -> a chi^2*D^2 intermediate = 10 GB  (impossible)
#   TEMPO chi_max 64  at N=7 -> 0.16 GB, K=10 in ~30 s, trace error ~1e-3  (fine)
if N == 4:
    K_heom, K_pi, HEOM_DEPTH, CHI_MAX = 25, 10, 3, 512
else:
    K_heom, K_pi, HEOM_DEPTH, CHI_MAX = 15, 10, 2, 64

K_SWEEP = [1, 5, 10, 15, 20, 25]   # windows scanned for the accuracy-vs-memory plot
overwrite = False                   # True -> write/overwrite data/run_N{N}.npz
RUNFILE = f'data/run_N{N}.npz'

res = {}
print(f'N = {N}  (D = {d*d})   K_heom = {K_heom}   K_pi = {K_pi}   '
      f'HEOM depth = {HEOM_DEPTH}   TEMPO chi_max = {CHI_MAX}')
print(f'target file: {RUNFILE}')

N = 4  (D = 16)   K_heom = 25   K_pi = 10   HEOM depth = 3   TEMPO chi_max = 128
target file: data/run_N4.npz


### Lindblad: ONE $16\times16$ map $\mathcal{L}(\Delta t)=e^{L\Delta t}\rightarrow M_k(\Delta t)$ → operator sum of dilated circuits

In [ ]:
t0 = time.time()

L_dt = lindblad_onestep_map(dt_fs, **MODEL)          # the whole classical cost
res['Lindblad'] = populations_kraus_qiskit(L_dt, d, rho0, n_steps)

print(f"Lindblad      {res['Lindblad']['n_qubits']} qubits, "
      f"{res['Lindblad']['n_circuits']} circuits, {time.time() - t0:.1f} s")

Lindblad      5 qubits, 1600 circuits, 1.5 s


### HEOM: ONE propagator $P=e^{\hat{\mathcal{L}}_{\rm HEOM}\Delta t}$ → short-time maps ($K$-window) → ONE dilated circuit

In [ ]:
t0 = time.time()

P, n_ado = heom_onestep_propagator(dt_fs, **MODEL, depth=HEOM_DEPTH, verbose=False)
maps_heom = heom_shorttime_maps(P, max(K_heom, max(K_SWEEP)), d)   # from P alone; deep enough for the sweep
res['HEOM'] = populations_memory_qiskit(maps_heom, rho0, K_heom, n_steps)

print(f"HEOM          {n_ado} ADOs, generator {P.shape[0]}x{P.shape[0]}, "
      f"{res['HEOM']['n_qubits']} qubits, {res['HEOM']['n_circuits']} circuits, "
      f"{time.time() - t0:.1f} s")

HEOM          165 ADOs, generator 2640x2640, 10 qubits, 76 circuits, 22.5 s


### Path integral: TEMPO short-time maps → ONE dilated circuit

In [ ]:
# The only method whose maps are genuinely expensive: the TEMPO/TNPI cost grows
# steeply with the memory window (N=4: K=10 -> 52 s, K=25 -> 474 s).
t0 = time.time()

pm = PathIntegralMap(H_FMO, dt_fs, K_pi, LAM_FMO, GAM_FMO, T_FMO, fs_to_cm=FS_TO_CM, kb_cm=KB_CM, eps=1e-6, chi_max=CHI_MAX)
maps_pi = np.array(pm.run(K_pi, verbose=False))
res['Path integral'] = populations_memory_qiskit(maps_pi, rho0, K_pi, n_steps)

print(f"Path integral {res['Path integral']['n_qubits']} qubits, "
      f"{res['Path integral']['n_circuits']} circuits, {time.time() - t0:.1f} s")

Path integral 9 qubits, 91 circuits, 21.8 s


### Benchmarks: the ordinary qutip solvers, no grid

They hand back the FULL density matrices, so the coherences have a reference too.

In [10]:
t0 = time.time()
bench_rho = {'HEOM':     standard_heom_rho(t_fs, rho0, **MODEL, depth=HEOM_DEPTH),  # qutip HEOMSolver
             'Lindblad': standard_lindblad_rho(t_fs, rho0, **MODEL)}                # qutip mesolve
bench_rho['Path integral'] = bench_rho['HEOM']    # no qutip path integral exists; HEOM is the exact ref
bench = {n: np.real(np.einsum('tii->ti', r)) for n, r in bench_rho.items()}
print(f'qutip benchmarks in {time.time() - t0:.1f} s')

print('\nmethod          qubits  circuits  ancilla |0>  vs qutip')
print('-------------------------------------------------------')
for n in ('Path integral', 'HEOM', 'Lindblad'):
    r = res[n]
    print(f"{n:15s} {r['n_qubits']:4d} {r['n_circuits']:9d}      "
          f"{r['p_success']:.3f}      {np.abs(r['pops'] - bench[n]).max():.2e}")

10.0%. Run time:   0.12s. Est. time left: 00:00:00:01
20.0%. Run time:   0.17s. Est. time left: 00:00:00:00
30.0%. Run time:   0.22s. Est. time left: 00:00:00:00
40.0%. Run time:   0.26s. Est. time left: 00:00:00:00
50.0%. Run time:   0.31s. Est. time left: 00:00:00:00
60.0%. Run time:   0.36s. Est. time left: 00:00:00:00
70.0%. Run time:   0.39s. Est. time left: 00:00:00:00
80.0%. Run time:   0.42s. Est. time left: 00:00:00:00
90.0%. Run time:   0.45s. Est. time left: 00:00:00:00
100.0%. Run time:   0.48s. Est. time left: 00:00:00:00
Total run time:   0.49s
qutip benchmarks in 0.6 s

method          qubits  circuits  ancilla |0>  vs qutip
-------------------------------------------------------
Path integral      9        91      0.469      1.10e-02
HEOM              10        76      0.482      2.37e-04
Lindblad           5      1600      0.019      9.29e-08


### The classical reference

The same propagation done as plain matrix algebra (`validation_and_helpers.ipynb`, no circuit). Agreement at $\sim10^{-15}$ says the dilation, the ancilla post-selection and the renormalisation bookkeeping inside the circuit are right — it does *not* say the circuit was skipped.

In [11]:
ref = {'Lindblad': populations_enhanced(L_dt, d, rho0, n_steps)['pops'],
       'HEOM': populations_enhanced_memory(maps_heom, rho0, K_heom, n_steps, via='direct')['pops'],
       'Path integral': populations_enhanced_memory(maps_pi, rho0, K_pi, n_steps, via='direct')['pops']}

for n in ('Path integral', 'HEOM', 'Lindblad'):
    print(f"  {n:15s} max|U^dag U - I| = {res[n]['unitarity']:.1e}   "
          f"circuits vs classical reference: {np.abs(res[n]['pops'] - ref[n]).max():.1e}")

  Path integral   max|U^dag U - I| = 5.6e-11   circuits vs classical reference: 3.7e-15
  HEOM            max|U^dag U - I| = 6.7e-12   circuits vs classical reference: 2.9e-15
  Lindblad        max|U^dag U - I| = 3.5e-11   circuits vs classical reference: 4.7e-15


### How much memory the register has to hold

Each method is swept only as deep as its own short-time maps go, and the $K=1$ curve (memory thrown away) is kept for the deviation plot.

In [12]:
t0 = time.time()
sources = {'Path integral': maps_pi, 'HEOM': maps_heom}
sweep = {}
for n, m in sources.items():
    ks = [k for k in K_SWEEP if k <= len(m) - 1]
    sweep[n] = (ks, [np.abs(populations_memory_qiskit(m, rho0, k, n_steps)['pops']
                            - bench[n]).max() for k in ks])
    print(f'  {n:15s} swept K = {ks}')

# path integral with the memory discarded, for the deviation plot
pops_K1 = populations_memory_qiskit(maps_pi, rho0, 1, n_steps)['pops']
print(f'sweep done in {time.time() - t0:.1f} s')

  Path integral   swept K = [1, 5, 10]
  HEOM            swept K = [1, 5, 10, 15, 20, 25]
sweep done in 28.6 s


### Store the run

Everything `results.ipynb` needs goes into one file. The Qiskit channel objects are dropped — a `DilatedChannel` carries the full $2^n\times2^n$ unitary — and `channel_of` rebuilds them from $E$ or the Kraus operators when a circuit has to be drawn.

In [13]:
if overwrite:
    for r in res.values():                     # too large to store, cheap to rebuild
        r.pop('channel', None)
        r.pop('channels', None)

    save_run(RUNFILE, dict(N=N, d=d, dt_fs=dt_fs, t_fs=t_fs, n_steps=n_steps,
                           K_heom=K_heom, K_pi=K_pi, heom_depth=HEOM_DEPTH,
                           chi_max=CHI_MAX, n_ado=n_ado,
                           res=res, bench_rho=bench_rho, ref=ref,
                           sweep=sweep, pops_K1=pops_K1))
    print(f'wrote {RUNFILE}  ({os.path.getsize(RUNFILE) / 1e6:.1f} MB)')
else:
    print('overwrite = False -> nothing written')

wrote data/run_N4.npz  (0.3 MB)
